# morphological_quantification_2026-01-02 — 04_marker_positive_region_review

**Feeds:** Fig 3f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 04 | Marker Positive-Region Review

This notebook defines one shared positivity standard per marker across the full dataset, then reviews the resulting positive regions on every image and z plane.

Aside from illumination correction, the thresholding procedure here is the same one used in `FOXF1_BMP4_3d_timelapse`.


## Cell Guide

- `Setup`: resolve the project root, import helper code, and define output paths.
- `Method Locked For This Run`: record the exact thresholding choices that are treated as fixed for this run.
- `Load Inputs`: load the per-z whole-morph geometry, consensus geometry, and posterior-click outputs from notebooks `02` and `03`.
- `Thresholding Procedure Used In This Run`: spell out the exact background-subtraction and pooled-threshold steps that are active right now.
- `Estimate Shared Marker Thresholds`: pool background-corrected within-morph pixels across all files and z planes, then fit one half-Gaussian threshold family per marker.
- `Review Global Threshold Fits`: inspect the pooled histograms, baseline fits, and candidate sigma cutoffs.
- `Representative Threshold-Boundary Pages`: compare `2σ / 3σ / 4σ / 5σ` positive-mask boundaries on one representative z plane per file.
- `Default Positive-Region Review Across Z`: inspect the default positive masks on every z plane for every file.
- `Quantify Default Threshold Stability Across Z`: summarize how the default positive area changes across z for each marker.
- `Inspect Most Atypical Across-Z Traces`: show the most suspicious across-z examples back in image space.
- `Save Stage Outputs`: write the thresholds, per-plane positive-region metrics, and positive-mask TIFFs for downstream notebooks.
- `Next Step`: use the reviewed marker-positive masks together with the posterior-oriented consensus axis for domain measurements.


In [ ]:
import json
import math
import sys
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().resolve().parent
elif (Path.cwd() / "notebooks").exists():
    ROOT = Path.cwd().resolve()
else:
    raise RuntimeError("Run this notebook from the project root or the notebooks/ directory.")

SCRIPTS_DIR = ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import morphology_quantification_helpers as mqh

pd.set_option("display.max_columns", 200)
plt.rcParams["figure.dpi"] = 120


## Method Locked For This Run

These are the fixed analysis choices for the current run of this notebook:

- Background subtraction: for each **file × z plane × marker**, subtract the median of that same plane's **off-cyst** pixels.
- Threshold fit: pool the resulting **background-corrected within-morph pixels** across the dataset and fit one shared `mu_bg` / `sigma_bg` per marker.
- Default threshold used downstream: `4σ` for all three markers.
  - `MESP2-mCherry = 4σ`
  - `FOXF1-YFP = 4σ`
  - `PAX8 = 4σ`
- Positive-mask cleanup: apply the same morphology cleanup on every plane after thresholding.
- Cohort filter: files with `include_in_analysis = False` in `results/manifests/analysis_manifest.tsv` are excluded.
- Current manual exclusion: file `31` is excluded from the active cohort.

Aside from illumination correction, this matches the thresholding procedure used in `FOXF1_BMP4_3d_timelapse`.


## Settings Notes

- Files flagged `include_in_analysis = False` in `results/manifests/analysis_manifest.tsv` are filtered out before threshold estimation and review.
- Marker display uses one marker-global absolute intensity window per channel, so brightness is comparable across files and z planes.
- This notebook does **not** choose a final signal-quantification z plane. It applies one shared marker standard to every z plane and saves the per-z outputs.
- Detailed off-cyst background-QC figures now live in `04a`, not in this notebook.
- If the boundary pages suggest a better sigma cutoff later, edit the defaults and rerun.


In [ ]:
ANALYSIS_MANIFEST_PATH = ROOT / "results" / "manifests" / "analysis_manifest.tsv"
CONSENSUS_TABLE_PATH = ROOT / "results" / "tables" / "whole_morph_consensus_geometry.tsv"
PER_Z_GEOMETRY_TABLE_PATH = ROOT / "results" / "tables" / "whole_morph_per_z_geometry.tsv"
POSTERIOR_PATH = ROOT / "results" / "annotations" / "manual_posterior_clicks.tsv"

THRESHOLD_TABLE_PATH = ROOT / "results" / "tables" / "04_global_marker_thresholds.tsv"
PLANE_METRICS_TABLE_PATH = ROOT / "results" / "tables" / "04_marker_positive_metrics_by_plane.tsv"
PARAMETERS_PATH = ROOT / "results" / "tables" / "04_marker_positive_region_parameters.json"
REVIEW_INPUT_TABLE_PATH = ROOT / "results" / "tables" / "04_marker_positive_review_input_by_plane.tsv"
DISPLAY_LIMITS_TABLE_PATH = ROOT / "results" / "tables" / "04_marker_display_limits.tsv"

POSITIVE_MASK_ROOT = ROOT / "results" / "masks" / "marker_positive_regions"
THRESHOLD_QC_DIR = ROOT / "results" / "qc" / "marker_threshold_review"
BOUNDARY_QC_DIR = ROOT / "results" / "qc" / "marker_threshold_boundary_review"
ALL_Z_QC_DIR = ROOT / "results" / "qc" / "marker_positive_region_all_z_review"

for path in [POSITIVE_MASK_ROOT, THRESHOLD_QC_DIR, BOUNDARY_QC_DIR, ALL_Z_QC_DIR]:
    path.mkdir(parents=True, exist_ok=True)

THRESHOLD_SIGMA_CHOICES = [2.0, 3.0, 4.0, 5.0]
DEFAULT_SIGMA_BY_MARKER = {
    "mesp2": 4.0,
    "foxf1": 4.0,
    "pax8": 4.0,
}

BACKGROUND_ESTIMATOR = "whole_off_morph"
ANNULUS_INNER_RADIUS_PX = 6
ANNULUS_OUTER_RADIUS_PX = 20
MIN_RING_PIXELS = 2000

if str(BACKGROUND_ESTIMATOR).strip().lower() in {"none", "raw", "no_subtraction", "global_raw_null"}:
    INTENSITY_AXIS_LABEL = "Raw within-morph intensity"
    ANALYSIS_IMAGE_LABEL = "raw marker image"
    DISPLAY_WINDOW_LABEL = "marker-global absolute raw-intensity display windows"
else:
    INTENSITY_AXIS_LABEL = "Background-corrected within-morph intensity"
    ANALYSIS_IMAGE_LABEL = "corrected marker image"
    DISPLAY_WINDOW_LABEL = "marker-global absolute corrected-intensity display windows"

POSITIVE_CLOSING_RADIUS = 2
POSITIVE_MIN_OBJECT_SIZE = 64
POSITIVE_HOLE_AREA = 32
BOUNDARY_DISPLAY_QUANTILES_BY_MARKER = {
    "mesp2": (0.01, 0.90),
    "foxf1": (0.01, 0.90),
    "pax8": (0.01, 0.995),
}

THRESHOLD_HIST_BINS = 256
THRESHOLD_SMOOTH_SIGMA_BINS = 2.0
THRESHOLD_HIST_QUANTILE_LOW = 0.001
THRESHOLD_HIST_QUANTILE_HIGH = 0.999
THRESHOLD_HIST_EDGE_PADDING_FRACTION = 0.05
SAMPLE_VALUES_PER_PLANE = 2048
THRESHOLD_RANDOM_SEED = 7

QUANTILE_LEVELS = np.array([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99], dtype=float)
QUANTILE_LABELS = [f"q{int(round(level * 100)):02d}" for level in QUANTILE_LEVELS]
CENTERED_QUANTILE_LABELS = [f"centered_{label}" for label in QUANTILE_LABELS]
QUANTILE_PERCENTILES = QUANTILE_LEVELS * 100.0
MARKER_PLOT_COLORS = {
    "mesp2": "tab:red",
    "foxf1": "goldenrod",
    "pax8": "tab:blue",
}

REPRESENTATIVE_ROWS_PER_PAGE = 4
ALL_Z_REVIEW_COLUMNS = 4
INLINE_BOUNDARY_EXAMPLES_PER_MARKER = 1
INLINE_ALL_Z_REVIEW_PAGES = 3
OPTIMIZATION_MODE = False


## Load Inputs


In [ ]:
manifest_df = pd.read_csv(ANALYSIS_MANIFEST_PATH, sep="\t")
included_file_paths = set(
    manifest_df.loc[
        manifest_df["include_in_analysis"].fillna(True).astype(bool),
        "file_path",
    ].astype(str)
)
excluded_markers_by_file_path = mqh.marker_exclusion_map_from_manifest(manifest_df)
excluded_marker_summary = [
    {
        "file_path": file_path,
        "excluded_marker_keys": ",".join(sorted(marker_keys)),
    }
    for file_path, marker_keys in sorted(excluded_markers_by_file_path.items())
]
excluded_marker_summary_df = pd.DataFrame(excluded_marker_summary)
consensus_df = pd.read_csv(CONSENSUS_TABLE_PATH, sep="\t")
consensus_df = consensus_df.loc[consensus_df["file_path"].astype(str).isin(included_file_paths)].copy()
per_z_geometry_df = pd.read_csv(PER_Z_GEOMETRY_TABLE_PATH, sep="\t")
per_z_geometry_df = per_z_geometry_df.loc[
    per_z_geometry_df["file_path"].astype(str).isin(included_file_paths)
].copy()
posterior_df = pd.read_csv(POSTERIOR_PATH, sep="\t")
posterior_df = posterior_df.loc[posterior_df["file_path"].astype(str).isin(included_file_paths)].copy()

display_plane_df = consensus_df.merge(
    per_z_geometry_df[
        [
            "file_path",
            "z_index",
            "mask_path",
            "acquisition_date",
            "acquisition_batch_label",
        ]
    ].rename(columns={"z_index": "display_z_index", "mask_path": "display_mask_path"}),
    on=["file_path", "display_z_index"],
    how="left",
)
if display_plane_df["display_mask_path"].isna().any():
    missing_ids = display_plane_df.loc[
        display_plane_df["display_mask_path"].isna(),
        "file_id",
    ].astype(int).tolist()
    raise RuntimeError(
        "Missing display-plane masks for file IDs: "
        + ", ".join(str(v) for v in missing_ids[:10])
    )

review_input_df = per_z_geometry_df.merge(
    consensus_df[
        [
            "file_path",
            "display_z_index",
            "retained_z_indices",
            "omitted_z_indices",
            "keep_only_z_indices",
            "n_retained_z",
            "n_total_z",
        ]
    ],
    on="file_path",
    how="left",
).merge(
    posterior_df[
        [
            "file_path",
            "posterior_click_x_px",
            "posterior_click_y_px",
        ]
    ],
    on="file_path",
    how="left",
)
review_input_df = review_input_df.sort_values(["file_id", "z_index"]).reset_index(drop=True)
review_input_df.to_csv(REVIEW_INPUT_TABLE_PATH, sep="\t", index=False)

print(f"Per-z geometry rows: {len(per_z_geometry_df)}")
print(f"Consensus files: {len(consensus_df)}")
print(f"Posterior clicks: {len(posterior_df)}")
print(f"Review input rows: {len(review_input_df)}")
if len(excluded_marker_summary_df):
    print("Per-file marker exclusions detected:")
    display(excluded_marker_summary_df)


## Thresholding Procedure Used In This Run

The active analysis path in this notebook is:

1. For each **file × z plane × marker**, take the raw marker image and the reviewed whole-morph mask from notebook `02`.
2. Measure the **off-cyst median** on that same z plane and subtract it from the full image for that marker.
3. After that subtraction, collect the **within-morph corrected pixels** from **all files and all z planes** for that marker.
4. Fit the pooled negative baseline from that corrected within-morph distribution:
   - `mu_bg`: the smoothed mode / baseline peak of the pooled histogram
   - `sigma_bg`: the left-half width of that same pooled histogram
5. Call positive pixels at `mu_bg + N*sigma_bg`, where the current default is `N = 4`.
6. Clean the positive masks morphologically in the same way on every plane.

So in short:

- local background subtraction uses **off-cyst pixels**
- the shared threshold fit uses **pooled corrected within-morph pixels**
- the same shared threshold family is then applied back to every z plane in the dataset

Aside from illumination correction, this is the same thresholding procedure used in `FOXF1_BMP4_3d_timelapse`.


## Estimate Shared Marker Thresholds


In [ ]:
threshold_df, threshold_diag = mqh.estimate_marker_thresholds_from_per_z_geometry(
    per_z_geometry_df=per_z_geometry_df,
    root=ROOT,
    excluded_markers_by_file_path=excluded_markers_by_file_path,
    threshold_sigma_choices=THRESHOLD_SIGMA_CHOICES,
    default_sigma_by_marker=DEFAULT_SIGMA_BY_MARKER,
    hist_bins=THRESHOLD_HIST_BINS,
    smooth_sigma_bins=THRESHOLD_SMOOTH_SIGMA_BINS,
    background_estimator=BACKGROUND_ESTIMATOR,
    annulus_inner_radius_px=ANNULUS_INNER_RADIUS_PX,
    annulus_outer_radius_px=ANNULUS_OUTER_RADIUS_PX,
    min_ring_pixels=MIN_RING_PIXELS,
    sample_values_per_plane=SAMPLE_VALUES_PER_PLANE,
    random_seed=THRESHOLD_RANDOM_SEED,
    hist_quantile_low=THRESHOLD_HIST_QUANTILE_LOW,
    hist_quantile_high=THRESHOLD_HIST_QUANTILE_HIGH,
    hist_edge_padding_fraction=THRESHOLD_HIST_EDGE_PADDING_FRACTION,
)
threshold_df.to_csv(THRESHOLD_TABLE_PATH, sep="\t", index=False)

marker_plane_metrics_df = mqh.quantify_marker_positive_regions(
    per_z_geometry_df=per_z_geometry_df,
    threshold_df=threshold_df,
    root=ROOT,
    positive_mask_root=POSITIVE_MASK_ROOT,
    excluded_markers_by_file_path=excluded_markers_by_file_path,
    background_estimator=BACKGROUND_ESTIMATOR,
    annulus_inner_radius_px=ANNULUS_INNER_RADIUS_PX,
    annulus_outer_radius_px=ANNULUS_OUTER_RADIUS_PX,
    min_ring_pixels=MIN_RING_PIXELS,
    closing_radius=POSITIVE_CLOSING_RADIUS,
    min_object_size=POSITIVE_MIN_OBJECT_SIZE,
    hole_area=POSITIVE_HOLE_AREA,
)
marker_plane_metrics_df.to_csv(PLANE_METRICS_TABLE_PATH, sep="\t", index=False)

parameters = {
    "threshold_sigma_choices": THRESHOLD_SIGMA_CHOICES,
    "default_sigma_by_marker": DEFAULT_SIGMA_BY_MARKER,
    "background_estimator": BACKGROUND_ESTIMATOR,
    "annulus_inner_radius_px": ANNULUS_INNER_RADIUS_PX,
    "annulus_outer_radius_px": ANNULUS_OUTER_RADIUS_PX,
    "min_ring_pixels": MIN_RING_PIXELS,
    "positive_closing_radius": POSITIVE_CLOSING_RADIUS,
    "positive_min_object_size": POSITIVE_MIN_OBJECT_SIZE,
    "positive_hole_area": POSITIVE_HOLE_AREA,
    "boundary_display_quantiles_by_marker": BOUNDARY_DISPLAY_QUANTILES_BY_MARKER,
    "threshold_hist_bins": THRESHOLD_HIST_BINS,
    "threshold_smooth_sigma_bins": THRESHOLD_SMOOTH_SIGMA_BINS,
    "threshold_hist_quantile_low": THRESHOLD_HIST_QUANTILE_LOW,
    "threshold_hist_quantile_high": THRESHOLD_HIST_QUANTILE_HIGH,
    "threshold_hist_edge_padding_fraction": THRESHOLD_HIST_EDGE_PADDING_FRACTION,
    "sample_values_per_plane": SAMPLE_VALUES_PER_PLANE,
    "threshold_random_seed": THRESHOLD_RANDOM_SEED,
    "notes": [
        "Thresholds are pooled across all files and all z planes from background-corrected within-morph intensities.",
        "Background correction uses the median of off-cyst pixels separately on each z plane.",
        "Positive masks are cleaned after thresholding using binary closing, small-object removal, and small-hole filling.",
        "This notebook applies the same marker thresholds to every image and z plane in the dataset.",
        "Representative marker images use marker-global absolute display windows derived once from pooled quantiles and then reused across images.",
        "Per-file marker exclusions from the analysis manifest are honored before threshold fitting and positive-mask quantification.",
    ],
}
PARAMETERS_PATH.write_text(json.dumps(parameters, indent=2) + "\n")

display_limit_rows = []
BOUNDARY_DISPLAY_LIMITS_BY_MARKER = {}
for marker_key in mqh.MARKER_KEYS:
    low_q, high_q = BOUNDARY_DISPLAY_QUANTILES_BY_MARKER.get(marker_key, (0.01, 0.995))
    corrected_vmin, corrected_vmax = mqh.corrected_display_limits_from_arrays(
        [threshold_diag[marker_key].get("sample_values", np.array([], dtype=np.float32))],
        low_q=float(low_q),
        high_q=float(high_q),
    )
    corrected_vmin = float(corrected_vmin)
    corrected_vmax = float(corrected_vmax)
    BOUNDARY_DISPLAY_LIMITS_BY_MARKER[marker_key] = (corrected_vmin, corrected_vmax)
    display_limit_rows.append(
        {
            "marker_key": marker_key,
            "marker_display_name": mqh.marker_display_name(marker_key),
            "display_low_quantile": float(low_q),
            "display_high_quantile": float(high_q),
            "display_vmin_intensity": corrected_vmin,
            "display_vmax_intensity": corrected_vmax,
        }
    )
display_limits_df = pd.DataFrame(display_limit_rows)
display_limits_df.to_csv(DISPLAY_LIMITS_TABLE_PATH, sep="\t", index=False)

threshold_display_df = threshold_df[
    [
        "marker_display_name",
        "sigma_multiple",
        "selected_as_default",
        "threshold_value",
        "baseline_location",
        "baseline_scale",
        "pooled_pixel_count",
        "included_plane_count",
        "pooled_background_median",
    ]
].copy()
display(threshold_display_df.style.hide(axis="index"))
display(display_limits_df.style.hide(axis="index"))

print(f"Wrote thresholds: {THRESHOLD_TABLE_PATH.relative_to(ROOT).as_posix()}")
print(f"Wrote per-plane marker metrics: {PLANE_METRICS_TABLE_PATH.relative_to(ROOT).as_posix()}")
print(f"Wrote parameters: {PARAMETERS_PATH.relative_to(ROOT).as_posix()}")
print(f"Wrote display limits: {DISPLAY_LIMITS_TABLE_PATH.relative_to(ROOT).as_posix()}")


## Review Global Threshold Fits

This section answers one question: does the pooled corrected within-morph histogram support a sensible shared `mu_bg + N*sigma_bg` threshold family for each marker?


In [ ]:
fig, axes = plt.subplots(
    2,
    len(mqh.MARKER_KEYS),
    figsize=(6.2 * len(mqh.MARKER_KEYS), 9.2),
    constrained_layout=True,
    squeeze=False,
)

for col_idx, marker_key in enumerate(mqh.MARKER_KEYS):
    diag = threshold_diag[marker_key]
    sample_values = np.asarray(diag.get("sample_values", np.array([], dtype=np.float32)), dtype=float)
    centers = np.asarray(diag["hist_centers"], dtype=float)
    counts = np.asarray(diag["hist_counts"], dtype=float)
    smooth = np.asarray(diag["smooth_counts"], dtype=float)
    fit_counts = np.asarray(diag["fit_counts"], dtype=float)
    marker_rows = threshold_df.loc[threshold_df["marker_key"] == marker_key].sort_values("sigma_multiple")
    default_row = marker_rows.loc[marker_rows["selected_as_default"].astype(bool)].iloc[0]
    baseline_location = float(diag["baseline_location"])
    baseline_scale = float(diag["baseline_scale"])
    threshold_value = float(default_row["threshold_value"])
    raw_mode_center = float(centers[int(np.nanargmax(counts))]) if np.any(counts > 0) else baseline_location
    bin_width = float(np.median(np.diff(centers))) if len(centers) > 1 else 1.0

    full_left_quantile = (
        float(np.quantile(sample_values, 0.001))
        if sample_values.size
        else raw_mode_center - 4.0 * baseline_scale
    )
    full_right_quantile = (
        float(np.quantile(sample_values, 0.9995))
        if sample_values.size
        else raw_mode_center + 8.0 * baseline_scale
    )
    context_xmin = full_left_quantile
    context_xmax = max(
        full_right_quantile,
        threshold_value + 0.75 * baseline_scale,
        baseline_location + 8.0 * baseline_scale,
    )
    context_span = context_xmax - context_xmin
    if np.isfinite(context_span) and context_span > 0:
        context_xmin -= 0.03 * context_span
        context_xmax += 0.03 * context_span
    if math.isclose(context_xmax, context_xmin):
        context_xmax = context_xmin + 1.0

    baseline_sample = sample_values[sample_values <= baseline_location] if sample_values.size else np.array([], dtype=float)
    left_quantile = (
        float(np.quantile(baseline_sample, 0.01))
        if baseline_sample.size
        else raw_mode_center - 4.0 * baseline_scale
    )
    right_quantile = (
        float(np.quantile(baseline_sample, 0.999))
        if baseline_sample.size
        else raw_mode_center + 4.0 * baseline_scale
    )
    zoom_half_width = max(
        raw_mode_center - left_quantile,
        right_quantile - raw_mode_center,
        abs(baseline_location - raw_mode_center),
        3.5 * baseline_scale,
    )
    zoom_half_width *= 1.05
    zoom_xmin = raw_mode_center - zoom_half_width
    zoom_xmax = max(
        raw_mode_center + zoom_half_width,
        threshold_value + 0.5 * baseline_scale,
        baseline_location + 4.5 * baseline_scale,
    )
    zoom_span = zoom_xmax - zoom_xmin
    if np.isfinite(zoom_span) and zoom_span > 0:
        zoom_xmin -= 0.02 * zoom_span
        zoom_xmax += 0.02 * zoom_span
    if math.isclose(zoom_xmax, zoom_xmin):
        zoom_xmax = zoom_xmin + 1.0

    ax = axes[0, col_idx]

    ax.plot(centers, counts, color="0.75", linewidth=1.0, label="pooled histogram")
    ax.plot(centers, smooth, color="black", linewidth=1.8, label="smoothed histogram")
    ax.plot(centers, fit_counts, color="#4c78a8", linewidth=1.4, linestyle="--", label="half-Gaussian fit")
    ax.axvline(float(diag["baseline_location"]), color="#2ca02c", linewidth=1.2, linestyle=":", label="baseline peak")

    for row in marker_rows.itertuples(index=False):
        color = "#d62728" if bool(row.selected_as_default) else "0.55"
        alpha = 0.95 if bool(row.selected_as_default) else 0.6
        ax.axvline(
            float(row.threshold_value),
            color=color,
            linewidth=1.4 if bool(row.selected_as_default) else 1.0,
            linestyle="-" if bool(row.selected_as_default) else "--",
            alpha=alpha,
        )
        ypos = 0.98 - 0.08 * list(marker_rows["sigma_multiple"]).index(float(row.sigma_multiple))
        ax.text(
            float(row.threshold_value),
            ypos * ax.get_ylim()[1],
            f"{float(row.sigma_multiple):g}σ",
            rotation=90,
            va="top",
            ha="right",
            fontsize=8,
            color=color,
        )

    ax.set_title(
        f"{mqh.marker_display_name(marker_key)}\n"
        f"Broader pooled-intensity context\n"
        f"peak={float(diag['baseline_location']):.1f}, sigma={float(diag['baseline_scale']):.1f}",
        fontsize=10,
    )
    ax.set_xlabel(INTENSITY_AXIS_LABEL)
    ax.set_ylabel("Pooled pixel count")
    ax.set_xlim(context_xmin, context_xmax)
    ax.grid(alpha=0.18)
    ax.text(
        0.03,
        0.97,
        (
            f"sample px={int(sample_values.size):,}\n"
            f"peak={baseline_location:.1f}\n"
            f"sigma={baseline_scale:.1f}\n"
            f"default={threshold_value:.1f}"
        ),
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8.2,
        bbox={"facecolor": "white", "alpha": 0.88, "edgecolor": "none", "pad": 2.0},
    )

    ax = axes[1, col_idx]
    ax.bar(
        centers,
        counts,
        width=bin_width,
        color="0.78",
        edgecolor="white",
        linewidth=0.2,
        align="center",
        label="raw histogram",
    )
    ax.plot(
        centers,
        smooth,
        color="black",
        linewidth=2.0,
        label=f"smoothed histogram (σ={float(diag['smooth_sigma_bins']):.1f} bins)",
    )
    ax.plot(
        centers,
        fit_counts,
        color="royalblue",
        linewidth=2.0,
        linestyle="--",
        label="half-Gaussian fit",
    )
    ax.axvline(
        raw_mode_center,
        color="tab:orange",
        linestyle="--",
        linewidth=1.8,
        label="highest raw-count bin",
    )
    ax.axvline(
        baseline_location,
        color="deepskyblue",
        linestyle="--",
        linewidth=2.0,
        label="baseline peak",
    )
    ax.axvline(
        baseline_location - baseline_scale,
        color="gold",
        linestyle=":",
        linewidth=1.6,
        label="peak +/- sigma",
    )
    ax.axvline(
        baseline_location + baseline_scale,
        color="gold",
        linestyle=":",
        linewidth=1.6,
    )
    ax.axvline(
        threshold_value,
        color="magenta",
        linewidth=2.0,
        alpha=0.92,
        label=f"default threshold ({float(default_row['sigma_multiple']):g}σ)",
    )
    ax.set_title(
        f"{mqh.marker_display_name(marker_key)}\nZoomed null-fit diagnostic",
        fontsize=10.2,
    )
    ax.set_xlabel(INTENSITY_AXIS_LABEL)
    ax.set_ylabel("Histogram count")
    ax.set_xlim(zoom_xmin, zoom_xmax)
    ax.grid(alpha=0.16)
    ax.legend(frameon=False, fontsize=7.8, loc="upper right")
    ax.text(
        0.03,
        0.97,
        (
            f"sample px={int(sample_values.size):,}\n"
            f"peak={baseline_location:.1f}\n"
            f"sigma={baseline_scale:.1f}\n"
            f"default={threshold_value:.1f}"
        ),
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8.2,
        bbox={"facecolor": "white", "alpha": 0.88, "edgecolor": "none", "pad": 2.0},
    )

handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, frameon=False, loc="upper center", ncol=min(4, len(handles)))
threshold_summary_path = THRESHOLD_QC_DIR / "04_global_marker_threshold_histograms.png"
fig.savefig(threshold_summary_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", threshold_summary_path.relative_to(ROOT).as_posix())


## Representative Threshold-Boundary Pages

This is the first image-based threshold sanity check: on one representative z plane per file, compare how the cleaned positive-mask boundary changes at `2σ`, `3σ`, `4σ`, and `5σ`.

The notebook shows **one inline example per marker** for quick review. Full page sets are written to `results/qc/marker_threshold_boundary_review/`.

These pages use one representative z plane per file (`display_z_index`) and overlay the cleaned positive-mask boundaries at `2σ`, `3σ`, `4σ`, and `5σ` on the same background-corrected marker image used for thresholding.


In [ ]:
@lru_cache(maxsize=2)
def load_cached_stack(file_path: str):
    stack = mqh.load_czi_stack(ROOT / str(file_path))
    channel_idx_map = mqh.marker_channel_index_map(stack.channels)
    dapi_idx = mqh.find_channel_index(stack.channels, ["dapi"])
    return stack, channel_idx_map, dapi_idx


def load_cached_plane_bundle(file_path: str, z_index: int) -> dict:
    stack, channel_idx_map, dapi_idx = load_cached_stack(str(file_path))
    z_idx = int(z_index)
    dapi = (
        np.asarray(stack.data_czyx[int(dapi_idx), z_idx], dtype=np.float32)
        if dapi_idx is not None
        else np.zeros(stack.data_czyx.shape[-2:], dtype=np.float32)
    )
    marker_planes = {}
    for marker_key in mqh.MARKER_KEYS:
        if not mqh.marker_is_available_for_file(file_path, marker_key, excluded_markers_by_file_path):
            marker_planes[marker_key] = None
            continue
        ch_idx = channel_idx_map.get(marker_key)
        marker_planes[marker_key] = (
            np.asarray(stack.data_czyx[int(ch_idx), z_idx], dtype=np.float32)
            if ch_idx is not None
            else None
        )
    return {
        "dapi": dapi,
        "mesp2": marker_planes["mesp2"],
        "foxf1": marker_planes["foxf1"],
        "pax8": marker_planes["pax8"],
        "overlay_rgb": mqh.build_overlay_rgb(
            dapi=dapi,
            mesp2=marker_planes["mesp2"],
            foxf1=marker_planes["foxf1"],
            pax8=marker_planes["pax8"],
        ),
    }


def corrected_marker_for_row(row, marker_key: str):
    plane = load_cached_plane_bundle(str(row.file_path), int(row.display_z_index))
    if plane[marker_key] is None:
        raise ValueError(
            f"Marker {marker_key!r} is excluded or unavailable for file {row.file_path!s}"
        )
    display_mask = mqh.load_binary_mask(ROOT / str(row.display_mask_path))
    corrected, background_value, background_mask = mqh.background_correct_signal(
        signal=plane[marker_key],
        organoid_mask=display_mask,
        background_estimator=BACKGROUND_ESTIMATOR,
        annulus_inner_radius_px=ANNULUS_INNER_RADIUS_PX,
        annulus_outer_radius_px=ANNULUS_OUTER_RADIUS_PX,
        min_ring_pixels=MIN_RING_PIXELS,
    )
    return plane, display_mask, corrected, float(background_value), background_mask


def threshold_value_for(marker_key: str, sigma_multiple: float) -> float:
    subset = threshold_df.loc[
        (threshold_df["marker_key"] == marker_key)
        & (np.isclose(threshold_df["sigma_multiple"].astype(float), float(sigma_multiple)))
    ]
    if subset.empty:
        raise KeyError(f"Missing threshold row for marker={marker_key!r}, sigma={sigma_multiple}")
    return float(subset["threshold_value"].iloc[0])


def render_boundary_pages_for_marker(marker_key: str) -> list[Path]:
    review_rows = display_plane_df.loc[
        display_plane_df["file_path"].map(
            lambda path: mqh.marker_is_available_for_file(path, marker_key, excluded_markers_by_file_path)
        )
    ].sort_values("file_id").reset_index(drop=True)
    if review_rows.empty:
        return []
    page_paths = []
    corrected_vmin, corrected_vmax = BOUNDARY_DISPLAY_LIMITS_BY_MARKER.get(marker_key, (0.0, 1.0))
    sigma_choices = [float(v) for v in THRESHOLD_SIGMA_CHOICES]
    threshold_map = {sigma: threshold_value_for(marker_key, sigma) for sigma in sigma_choices}

    for page_idx, page_rows in enumerate(mqh.chunked(review_rows.itertuples(index=False), REPRESENTATIVE_ROWS_PER_PAGE), start=1):
        n_rows = len(page_rows)
        n_cols = 1 + len(sigma_choices)
        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(3.15 * n_cols, 3.05 * n_rows),
            constrained_layout=True,
        )
        axes = np.asarray(axes)
        if axes.ndim == 1:
            axes = axes[None, :]

        for row_idx, row in enumerate(page_rows):
            plane, display_mask, corrected, background_value, _ = corrected_marker_for_row(row, marker_key)

            ax0 = axes[row_idx, 0]
            ax0.imshow(mqh.robust_rescale(plane["dapi"]), cmap="gray", interpolation="nearest")
            mqh.plot_mask_outline(ax0, display_mask, color="yellow", linewidth=1.0)
            ax0.set_title(
                f"{int(row.file_id):02d} | z={int(row.display_z_index)}\nDAPI + morph",
                fontsize=9,
            )
            ax0.set_xticks([])
            ax0.set_yticks([])

            for col_idx, sigma_value in enumerate(sigma_choices, start=1):
                ax = axes[row_idx, col_idx]
                ax.imshow(
                    corrected,
                    cmap="magma",
                    vmin=corrected_vmin,
                    vmax=corrected_vmax,
                    interpolation="nearest",
                )
                mqh.plot_mask_outline(ax, display_mask, color="white", linewidth=0.9)
                positive_mask = mqh.positive_mask_from_corrected(
                    corrected=corrected,
                    organoid_mask=display_mask,
                    threshold_value=threshold_map[sigma_value],
                    closing_radius=POSITIVE_CLOSING_RADIUS,
                    min_object_size=POSITIVE_MIN_OBJECT_SIZE,
                    hole_area=POSITIVE_HOLE_AREA,
                )
                mqh.plot_mask_outline(
                    ax,
                    positive_mask,
                    color=mqh.MARKER_OUTLINE_COLORS[marker_key],
                    linewidth=1.2,
                )
                positive_pixels = int(np.sum(positive_mask))
                ax.set_title(
                    f"{sigma_value:g}σ | +px={positive_pixels}",
                    fontsize=9,
                )
                ax.set_xticks([])
                ax.set_yticks([])

            if np.isfinite(background_value):
                figure_label = f"{int(row.file_id):02d} | bg={background_value:.1f}"
            else:
                figure_label = f"{int(row.file_id):02d}"
            axes[row_idx, 0].text(
                0.02,
                0.02,
                figure_label,
                transform=axes[row_idx, 0].transAxes,
                ha="left",
                va="bottom",
                fontsize=8,
                color="white",
                bbox={"facecolor": "black", "alpha": 0.55, "pad": 2},
            )

        fig.suptitle(
            f"{mqh.marker_display_name(marker_key)} threshold-boundary review on representative z planes",
            fontsize=11.5,
        )
        page_path = BOUNDARY_QC_DIR / f"{marker_key}_threshold_boundary_page{page_idx:02d}.png"
        fig.savefig(page_path, dpi=180, bbox_inches="tight")
        plt.close(fig)
        page_paths.append(page_path)
    return page_paths


def choose_boundary_inline_example_row(marker_key: str):
    default_sigma = float(DEFAULT_SIGMA_BY_MARKER.get(marker_key, 4.0))
    review_sub = marker_plane_metrics_df.loc[
        (marker_plane_metrics_df["marker_key"] == marker_key)
        & np.isclose(marker_plane_metrics_df["threshold_sigma_multiple"].astype(float), default_sigma)
    ].copy()
    review_sub = review_sub.merge(
        display_plane_df[["file_path", "display_z_index"]],
        on="file_path",
        how="inner",
    )
    review_sub = review_sub.loc[
        review_sub["z_index"].astype(int) == review_sub["display_z_index"].astype(int)
    ].copy()
    review_sub = review_sub.sort_values(["file_id", "z_index"]).reset_index(drop=True)
    if review_sub.empty:
        return None
    median_fraction = float(review_sub["positive_fraction"].median())
    review_sub["distance_from_median_fraction"] = np.abs(
        review_sub["positive_fraction"].astype(float) - median_fraction
    )
    review_sub["distance_from_default_sigma"] = np.abs(
        review_sub["threshold_sigma_multiple"].astype(float) - default_sigma
    )
    ranked = review_sub.sort_values(
        ["distance_from_default_sigma", "distance_from_median_fraction", "file_id"]
    ).reset_index(drop=True)
    selected = ranked.iloc[0]
    return display_plane_df.loc[
        (display_plane_df["file_path"].astype(str) == str(selected["file_path"]))
        & (display_plane_df["display_z_index"].astype(int) == int(selected["z_index"]))
    ].iloc[0]


def render_boundary_inline_example(marker_key: str) -> Path | None:
    row = choose_boundary_inline_example_row(marker_key)
    if row is None:
        return None

    corrected_vmin, corrected_vmax = BOUNDARY_DISPLAY_LIMITS_BY_MARKER.get(marker_key, (0.0, 1.0))
    sigma_choices = [float(v) for v in THRESHOLD_SIGMA_CHOICES]
    threshold_map = {sigma: threshold_value_for(marker_key, sigma) for sigma in sigma_choices}
    plane, display_mask, corrected, background_value, _ = corrected_marker_for_row(row, marker_key)

    fig, axes = plt.subplots(
        1,
        1 + len(sigma_choices),
        figsize=(3.15 * (1 + len(sigma_choices)), 3.2),
        constrained_layout=True,
        squeeze=False,
    )
    axes = axes.ravel()

    ax0 = axes[0]
    ax0.imshow(mqh.robust_rescale(plane["dapi"]), cmap="gray", interpolation="nearest")
    mqh.plot_mask_outline(ax0, display_mask, color="yellow", linewidth=1.0)
    ax0.set_title(
        f"{int(row.file_id):02d} | z={int(row.display_z_index)}\nDAPI + morph",
        fontsize=9,
    )
    ax0.set_xticks([])
    ax0.set_yticks([])

    for col_idx, sigma_value in enumerate(sigma_choices, start=1):
        ax = axes[col_idx]
        ax.imshow(
            corrected,
            cmap="magma",
            vmin=corrected_vmin,
            vmax=corrected_vmax,
            interpolation="nearest",
        )
        mqh.plot_mask_outline(ax, display_mask, color="white", linewidth=0.9)
        positive_mask = mqh.positive_mask_from_corrected(
            corrected=corrected,
            organoid_mask=display_mask,
            threshold_value=threshold_map[sigma_value],
            closing_radius=POSITIVE_CLOSING_RADIUS,
            min_object_size=POSITIVE_MIN_OBJECT_SIZE,
            hole_area=POSITIVE_HOLE_AREA,
        )
        mqh.plot_mask_outline(
            ax,
            positive_mask,
            color=mqh.MARKER_OUTLINE_COLORS[marker_key],
            linewidth=1.2,
        )
        positive_pixels = int(np.sum(positive_mask))
        ax.set_title(f"{sigma_value:g}σ | +px={positive_pixels}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

    if np.isfinite(background_value):
        figure_label = f"{int(row.file_id):02d} | bg={background_value:.1f}"
    else:
        figure_label = f"{int(row.file_id):02d}"
    axes[0].text(
        0.02,
        0.02,
        figure_label,
        transform=axes[0].transAxes,
        ha="left",
        va="bottom",
        fontsize=8,
        color="white",
        bbox={"facecolor": "black", "alpha": 0.55, "pad": 2},
    )

    fig.suptitle(
        f"{mqh.marker_display_name(marker_key)} representative threshold-boundary example",
        fontsize=11.2,
    )
    out_path = BOUNDARY_QC_DIR / f"{marker_key}_threshold_boundary_inline_example.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return out_path


In [ ]:
boundary_page_paths = {}
for marker_key in mqh.MARKER_KEYS:
    paths = render_boundary_pages_for_marker(marker_key)
    boundary_page_paths[marker_key] = paths
    print(
        f"{mqh.marker_display_name(marker_key)} boundary pages: "
        f"{len(paths)} written to {BOUNDARY_QC_DIR.relative_to(ROOT).as_posix()}"
    )
    if not OPTIMIZATION_MODE:
        inline_example_path = render_boundary_inline_example(marker_key)
        if inline_example_path is not None:
            display(Markdown(f"### {mqh.marker_display_name(marker_key)}"))
            display(Image(filename=str(inline_example_path)))


## Default Positive-Region Review Across Z

This is the main image review section for the locked default thresholds.

The notebook shows **three inline file-level pages** for quick review. Full page sets are written to `results/qc/marker_positive_region_all_z_review/`.

These pages overlay the default marker-positive masks from the selected default sigma values on **every z plane** for each file.

The marker intensities here use the same marker-global absolute display windows as `Representative Threshold-Boundary Pages`, so background/foreground contrast is comparable across files and z planes.


In [ ]:
metrics_lookup = {
    (str(row.file_path), int(row.z_index), str(row.marker_key)): row
    for row in marker_plane_metrics_df.itertuples(index=False)
}


def scaled_corrected_marker(corrected: np.ndarray, marker_key: str) -> np.ndarray:
    vmin, vmax = BOUNDARY_DISPLAY_LIMITS_BY_MARKER.get(marker_key, (0.0, 1.0))
    vmin = float(vmin)
    vmax = float(vmax)
    if not np.isfinite(vmin):
        vmin = float(np.nanmin(corrected)) if np.isfinite(corrected).any() else 0.0
    if not np.isfinite(vmax) or vmax <= vmin:
        vmax = vmin + 1.0
    scaled = (np.asarray(corrected, dtype=np.float32) - vmin) / (vmax - vmin)
    return np.clip(scaled, 0.0, 1.0)


def build_all_z_review_rgb(plane: dict, organoid_mask: np.ndarray) -> np.ndarray:
    dapi_gray = mqh.robust_rescale(plane["dapi"])
    marker_rgb = np.zeros((*dapi_gray.shape, 3), dtype=np.float32)
    for channel_idx, marker_key in enumerate(mqh.MARKER_KEYS):
        if plane[marker_key] is None:
            continue
        corrected, _, _ = mqh.background_correct_signal(
            signal=plane[marker_key],
            organoid_mask=organoid_mask,
            background_estimator=BACKGROUND_ESTIMATOR,
            annulus_inner_radius_px=ANNULUS_INNER_RADIUS_PX,
            annulus_outer_radius_px=ANNULUS_OUTER_RADIUS_PX,
            min_ring_pixels=MIN_RING_PIXELS,
        )
        marker_rgb[..., channel_idx] = scaled_corrected_marker(corrected, marker_key)

    base = np.stack([dapi_gray, dapi_gray, dapi_gray], axis=-1)
    return np.clip(0.35 * base + 0.85 * marker_rgb, 0.0, 1.0)


def render_all_z_positive_review_pages() -> list[Path]:
    page_paths = []
    for row in display_plane_df.sort_values("file_id").itertuples(index=False):
        file_sub = review_input_df.loc[
            review_input_df["file_path"].astype(str) == str(row.file_path)
        ].sort_values("z_index")
        n_planes = len(file_sub)
        n_cols = int(ALL_Z_REVIEW_COLUMNS)
        n_rows = int(math.ceil(n_planes / n_cols))
        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(4.0 * n_cols, 4.0 * n_rows),
            constrained_layout=True,
        )
        axes = np.asarray(axes).reshape(n_rows, n_cols)
        flat_axes = axes.ravel()

        for ax in flat_axes[n_planes:]:
            ax.axis("off")

        for ax, plane_row in zip(flat_axes, file_sub.itertuples(index=False)):
            plane = load_cached_plane_bundle(str(plane_row.file_path), int(plane_row.z_index))
            mask = mqh.load_binary_mask(ROOT / str(plane_row.mask_path))
            ax.imshow(build_all_z_review_rgb(plane, mask), interpolation="nearest")
            mqh.plot_mask_outline(ax, mask, color="white", linewidth=0.9)

            label_bits = [f"z={int(plane_row.z_index):02d}"]
            if int(plane_row.z_index) == int(row.display_z_index):
                label_bits.append("display")

            for marker_key in mqh.MARKER_KEYS:
                metric_row = metrics_lookup.get((str(plane_row.file_path), int(plane_row.z_index), marker_key))
                if metric_row is None:
                    continue
                positive_mask = mqh.load_binary_mask(ROOT / str(metric_row.positive_mask_path))
                mqh.plot_mask_outline(
                    ax,
                    positive_mask,
                    color=mqh.MARKER_OUTLINE_COLORS[marker_key],
                    linewidth=1.0,
                )
                label_bits.append(f"{marker_key[0].upper()}={int(metric_row.positive_pixels)}")

            ax.set_title(" | ".join(label_bits), fontsize=8.5)
            ax.set_xticks([])
            ax.set_yticks([])

        fig.suptitle(
            f"{int(row.file_id):02d} | default marker-positive masks across z",
            fontsize=11.2,
        )
        page_path = ALL_Z_QC_DIR / f"{int(row.file_id):02d}_all_z_marker_positive_review.png"
        fig.savefig(page_path, dpi=180, bbox_inches="tight")
        plt.close(fig)
        page_paths.append(page_path)
    return page_paths


def choose_inline_all_z_review_paths(page_paths: list[Path], n_examples: int) -> list[Path]:
    if not page_paths or n_examples <= 0:
        return []
    if len(page_paths) <= n_examples:
        return list(page_paths)
    idx_values = np.linspace(0, len(page_paths) - 1, n_examples)
    chosen_indices = []
    for idx in idx_values:
        rounded = int(round(float(idx)))
        if rounded not in chosen_indices:
            chosen_indices.append(rounded)
    if len(chosen_indices) < n_examples:
        for idx in range(len(page_paths)):
            if idx not in chosen_indices:
                chosen_indices.append(idx)
            if len(chosen_indices) >= n_examples:
                break
    return [page_paths[idx] for idx in chosen_indices[:n_examples]]


In [ ]:
all_z_review_paths = render_all_z_positive_review_pages()
print(
    f"All-z default positive-region review pages: {len(all_z_review_paths)} "
    f"written to {ALL_Z_QC_DIR.relative_to(ROOT).as_posix()}"
)
if not OPTIMIZATION_MODE:
    for path in choose_inline_all_z_review_paths(all_z_review_paths, INLINE_ALL_Z_REVIEW_PAGES):
        display(Image(filename=str(path)))


## Quantify Default Threshold Stability Across Z

This section asks whether the locked default thresholds behave stably through the z stack, or whether positive area rises and falls sharply with focus.

Here we summarize how much the default positive-pixel area changes across z for each file and marker.

The main visual is the within-file positive area normalized to that file's own peak positive area:
- if thresholded area is strongly focus-dependent, traces should rise toward a peak z and fall away from it
- if out-of-focus bleed keeps thresholded area high, traces may stay broad and comparatively flat


In [ ]:
Z_STABILITY_QC_DIR = ROOT / "results" / "qc" / "marker_z_stability_review"
Z_STABILITY_QC_DIR.mkdir(parents=True, exist_ok=True)
STABILITY_TABLE_PATH = ROOT / "results" / "tables" / "04_marker_z_stability_by_file.tsv"


def summarize_marker_z_stability(metrics_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (file_id, file_path, marker_key), sub in metrics_df.groupby(
        ["file_id", "file_path", "marker_key"], sort=True
    ):
        sub = sub.sort_values("z_index").reset_index(drop=True)
        z_values = sub["z_index"].astype(int).to_numpy()
        positive_pixels = sub["positive_pixels"].astype(float).to_numpy()
        positive_fraction = sub["positive_fraction"].astype(float).to_numpy()
        if positive_pixels.size == 0:
            continue

        peak_idx = int(np.argmax(positive_pixels))
        peak_pixels = float(positive_pixels[peak_idx])
        normalized = (
            positive_pixels / peak_pixels
            if np.isfinite(peak_pixels) and peak_pixels > 0
            else np.full_like(positive_pixels, np.nan, dtype=float)
        )
        rel_z = (
            np.linspace(0.0, 1.0, positive_pixels.size)
            if positive_pixels.size > 1
            else np.array([0.5], dtype=float)
        )
        edge_values = (
            np.array([normalized[0], normalized[-1]], dtype=float)
            if normalized.size > 1
            else np.array([normalized[0]], dtype=float)
        )
        edge_to_peak_ratio = float(np.nanmean(edge_values))
        peak_center_offset = (
            abs(float(rel_z[peak_idx]) - 0.5)
            if rel_z.size > 0
            else float("nan")
        )
        cv_positive_pixels = (
            float(np.nanstd(positive_pixels) / np.nanmean(positive_pixels))
            if np.nanmean(positive_pixels) > 0
            else float("nan")
        )
        range_ratio = (
            float((np.nanmax(positive_pixels) - np.nanmin(positive_pixels)) / np.nanmax(positive_pixels))
            if np.nanmax(positive_pixels) > 0
            else float("nan")
        )

        rows.append(
            {
                "file_id": int(file_id),
                "file_path": str(file_path),
                "marker_key": str(marker_key),
                "z_count": int(len(sub)),
                "peak_z_index": int(z_values[peak_idx]),
                "peak_positive_pixels": peak_pixels,
                "median_positive_pixels": float(np.nanmedian(positive_pixels)),
                "median_positive_fraction": float(np.nanmedian(positive_fraction)),
                "edge_to_peak_ratio": edge_to_peak_ratio,
                "cv_positive_pixels": cv_positive_pixels,
                "range_ratio": range_ratio,
                "peak_center_offset": peak_center_offset,
                "relative_z_profile": json.dumps(
                    [
                        {
                            "z_index": int(z),
                            "relative_z": float(rz),
                            "positive_pixels": float(px),
                            "normalized_positive_pixels": float(npx),
                        }
                        for z, rz, px, npx in zip(z_values, rel_z, positive_pixels, normalized)
                    ]
                ),
            }
        )
    return pd.DataFrame(rows).sort_values(["marker_key", "file_id"]).reset_index(drop=True)


stability_df = summarize_marker_z_stability(marker_plane_metrics_df)
stability_df.to_csv(STABILITY_TABLE_PATH, sep="\t", index=False)
print("Wrote z-stability table:", STABILITY_TABLE_PATH.relative_to(ROOT).as_posix())
for marker_key in mqh.MARKER_KEYS:
    sub = stability_df.loc[stability_df["marker_key"] == marker_key]
    print(
        f"{mqh.marker_display_name(marker_key)}: "
        f"median edge/peak={float(np.nanmedian(sub['edge_to_peak_ratio'])):.2f}, "
        f"median CV={float(np.nanmedian(sub['cv_positive_pixels'])):.2f}, "
        f"median range={float(np.nanmedian(sub['range_ratio'])):.2f}"
    )


In [ ]:
def render_marker_z_stability_summary(stability_df: pd.DataFrame) -> Path:
    grid = np.linspace(0.0, 1.0, 101)
    fig, axes = plt.subplots(
        1,
        len(mqh.MARKER_KEYS),
        figsize=(4.7 * len(mqh.MARKER_KEYS), 4.4),
        constrained_layout=True,
    )
    axes = np.atleast_1d(np.asarray(axes)).ravel()

    color_map = {
        "mesp2": "#d62728",
        "foxf1": "#2ca02c",
        "pax8": "#1f77b4",
    }

    for col_idx, marker_key in enumerate(mqh.MARKER_KEYS):
        sub = stability_df.loc[stability_df["marker_key"] == marker_key].copy()
        profiles = []
        ax = axes[col_idx]
        for row in sub.itertuples(index=False):
            profile_df = pd.DataFrame(json.loads(row.relative_z_profile))
            x = profile_df["relative_z"].astype(float).to_numpy()
            y = profile_df["normalized_positive_pixels"].astype(float).to_numpy()
            if len(x) == 1:
                interp = np.full_like(grid, y[0], dtype=float)
            else:
                interp = np.interp(grid, x, y)
            profiles.append(interp)
            ax.plot(x, y, color="0.75", linewidth=0.9, alpha=0.8)

        if profiles:
            profile_stack = np.vstack(profiles)
            median_profile = np.nanmedian(profile_stack, axis=0)
            q25 = np.nanquantile(profile_stack, 0.25, axis=0)
            q75 = np.nanquantile(profile_stack, 0.75, axis=0)
            ax.fill_between(grid, q25, q75, color=color_map[marker_key], alpha=0.18)
            ax.plot(grid, median_profile, color=color_map[marker_key], linewidth=2.4)

        ax.set_title(f"{mqh.marker_display_name(marker_key)}\nNormalized positive area across z", fontsize=10.2)
        ax.set_xlabel("Relative z position within stack")
        ax.set_ylabel("Positive area / within-file max")
        ax.set_xlim(0.0, 1.0)
        ax.set_ylim(0.0, 1.05)
        ax.grid(alpha=0.2)
        median_edge = float(np.nanmedian(sub["edge_to_peak_ratio"].astype(float).to_numpy()))
        ax.text(
            0.03,
            0.97,
            (
                f"n={len(sub)}\n"
                f"median edge/peak={median_edge:.2f}\n"
                f"median CV={float(np.nanmedian(sub['cv_positive_pixels'])):.2f}\n"
                f"median range={float(np.nanmedian(sub['range_ratio'])):.2f}"
            ),
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8.2,
            bbox={"facecolor": "white", "alpha": 0.88, "edgecolor": "none", "pad": 2.0},
        )

    out_path = Z_STABILITY_QC_DIR / "04_marker_z_stability_summary.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return out_path


z_stability_summary_path = render_marker_z_stability_summary(stability_df)
print("Z-stability summary figure:", z_stability_summary_path.relative_to(ROOT).as_posix())
if not OPTIMIZATION_MODE:
    display(Image(filename=str(z_stability_summary_path)))


## Inspect Most Atypical Across-Z Traces

This section pulls the most suspicious across-z behavior from the summary traces above and shows those files back in image space.

These pages show the most atypical across-z stability examples for the markers that looked most suspicious in the cohort-level stability traces.

They use the same marker-global absolute display windows as `Representative Threshold-Boundary Pages`.


In [ ]:
OUTLIER_REVIEW_MARKERS = ["foxf1", "pax8"]


def interpolate_relative_profile(row, grid: np.ndarray) -> np.ndarray:
    profile_df = pd.DataFrame(json.loads(row.relative_z_profile))
    x = profile_df["relative_z"].astype(float).to_numpy()
    y = profile_df["normalized_positive_pixels"].astype(float).to_numpy()
    if len(x) == 1:
        return np.full_like(grid, y[0], dtype=float)
    return np.interp(grid, x, y)


def identify_top_outlier_rows(stability_df: pd.DataFrame, marker_keys: list[str]) -> pd.DataFrame:
    grid = np.linspace(0.0, 1.0, 101)
    outlier_rows = []
    for marker_key in marker_keys:
        sub = stability_df.loc[stability_df["marker_key"] == marker_key].copy()
        if sub.empty:
            continue
        profiles = []
        for row in sub.itertuples(index=False):
            profiles.append(interpolate_relative_profile(row, grid))
        profile_stack = np.vstack(profiles)
        median_profile = np.nanmedian(profile_stack, axis=0)
        distances = np.nanmean(np.abs(profile_stack - median_profile[None, :]), axis=1)
        sub = sub.reset_index(drop=True)
        sub["profile_l1_to_median"] = distances
        outlier_rows.append(
            sub.sort_values(["profile_l1_to_median", "range_ratio", "cv_positive_pixels"], ascending=[False, False, False]).iloc[0]
        )
    if not outlier_rows:
        return pd.DataFrame()
    return pd.DataFrame(outlier_rows).sort_values(["marker_key", "file_id"]).reset_index(drop=True)


def render_outlier_across_z_page(file_path: str, marker_key: str, outlier_score: float) -> Path:
    file_sub = review_input_df.loc[
        review_input_df["file_path"].astype(str) == str(file_path)
    ].sort_values("z_index")
    if file_sub.empty:
        raise RuntimeError(f"Missing review-input rows for {file_path}")

    file_id = int(file_sub["file_id"].iloc[0])
    n_rows = len(file_sub)
    n_cols = 1 + len(mqh.MARKER_KEYS)
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(3.2 * n_cols, 2.9 * n_rows),
        constrained_layout=True,
    )
    axes = np.asarray(axes)
    if axes.ndim == 1:
        axes = axes[None, :]

    for row_idx, plane_row in enumerate(file_sub.itertuples(index=False)):
        plane = load_cached_plane_bundle(str(plane_row.file_path), int(plane_row.z_index))
        mask = mqh.load_binary_mask(ROOT / str(plane_row.mask_path))

        ax0 = axes[row_idx, 0]
        ax0.imshow(mqh.robust_rescale(plane["dapi"]), cmap="gray", interpolation="nearest")
        mqh.plot_mask_outline(ax0, mask, color="white", linewidth=0.9)
        ax0.set_title(f"z={int(plane_row.z_index):02d} | DAPI + morph", fontsize=8.8)
        ax0.set_xticks([])
        ax0.set_yticks([])

        for col_idx, current_marker_key in enumerate(mqh.MARKER_KEYS, start=1):
            ax = axes[row_idx, col_idx]
            if plane[current_marker_key] is None:
                ax.imshow(
                    np.full(mask.shape, 0.18, dtype=np.float32),
                    cmap="gray",
                    vmin=0.0,
                    vmax=1.0,
                    interpolation="nearest",
                )
                mqh.plot_mask_outline(ax, mask, color="white", linewidth=0.9)
                title_prefix = "OUTLIER | " if current_marker_key == marker_key else ""
                ax.set_title(
                    f"{title_prefix}{mqh.marker_display_name(current_marker_key)}\nexcluded",
                    fontsize=8.4,
                )
                ax.set_xticks([])
                ax.set_yticks([])
                continue
            corrected, _, _ = mqh.background_correct_signal(
                signal=plane[current_marker_key],
                organoid_mask=mask,
                background_estimator=BACKGROUND_ESTIMATOR,
                annulus_inner_radius_px=ANNULUS_INNER_RADIUS_PX,
                annulus_outer_radius_px=ANNULUS_OUTER_RADIUS_PX,
                min_ring_pixels=MIN_RING_PIXELS,
            )
            corrected_vmin, corrected_vmax = BOUNDARY_DISPLAY_LIMITS_BY_MARKER.get(current_marker_key, (0.0, 1.0))
            ax.imshow(
                corrected,
                cmap="magma",
                vmin=float(corrected_vmin),
                vmax=float(corrected_vmax),
                interpolation="nearest",
            )
            mqh.plot_mask_outline(ax, mask, color="white", linewidth=0.9)
            metric_row = metrics_lookup.get((str(plane_row.file_path), int(plane_row.z_index), current_marker_key))
            positive_pixels = 0
            if metric_row is not None:
                positive_mask = mqh.load_binary_mask(ROOT / str(metric_row.positive_mask_path))
                mqh.plot_mask_outline(
                    ax,
                    positive_mask,
                    color=mqh.MARKER_OUTLINE_COLORS[current_marker_key],
                    linewidth=1.1 if current_marker_key == marker_key else 0.95,
                )
                positive_pixels = int(metric_row.positive_pixels)
            title_prefix = "OUTLIER | " if current_marker_key == marker_key else ""
            ax.set_title(
                f"{title_prefix}{mqh.marker_display_name(current_marker_key)}\n+px={positive_pixels}",
                fontsize=8.4,
            )
            ax.set_xticks([])
            ax.set_yticks([])

    fig.suptitle(
        f"{file_id:02d} | {mqh.marker_display_name(marker_key)} atypical across-z trace | score={float(outlier_score):.3f}",
        fontsize=11.2,
    )
    out_path = Z_STABILITY_QC_DIR / f"{file_id:02d}_{marker_key}_outlier_fixed_global_marker_contrast.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return out_path


outlier_review_df = identify_top_outlier_rows(stability_df, OUTLIER_REVIEW_MARKERS)
outlier_review_paths = []
if outlier_review_df.empty:
    print("No across-z outlier review pages were generated.")
else:
    for row in outlier_review_df.itertuples(index=False):
        out_path = render_outlier_across_z_page(
            file_path=str(row.file_path),
            marker_key=str(row.marker_key),
            outlier_score=float(row.profile_l1_to_median),
        )
        outlier_review_paths.append(out_path)
        print(
            f"{mqh.marker_display_name(str(row.marker_key))} outlier review: "
            f"{out_path.relative_to(ROOT).as_posix()}"
        )
        if not OPTIMIZATION_MODE:
            display(Image(filename=str(out_path)))


## Save Stage Outputs


In [ ]:
created_paths = [
    THRESHOLD_TABLE_PATH,
    PLANE_METRICS_TABLE_PATH,
    STABILITY_TABLE_PATH,
    PARAMETERS_PATH,
    REVIEW_INPUT_TABLE_PATH,
    THRESHOLD_QC_DIR / "04_global_marker_threshold_histograms.png",
    z_stability_summary_path,
]
if "outlier_review_paths" in globals():
    created_paths.extend(outlier_review_paths)

print("Created files")
for path in created_paths:
    if path.exists():
        print("-", path.relative_to(ROOT).as_posix())

print("\nMarker positive-mask root")
print("-", POSITIVE_MASK_ROOT.relative_to(ROOT).as_posix())

print("\nQC directories")
print("-", BOUNDARY_QC_DIR.relative_to(ROOT).as_posix())
print("-", ALL_Z_QC_DIR.relative_to(ROOT).as_posix())
print("-", Z_STABILITY_QC_DIR.relative_to(ROOT).as_posix())


## Next Step

Use the reviewed marker-positive masks together with the consensus axis and posterior orientation to quantify marker domain size, bilaterality, and position along the trunk-morph axis.
